**MODELING**

In [7]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
import seaborn as sns
import ast

In [8]:
label_to_emotion = {
    0: "amusement",
    1: "excitement",
    2: "joy",
    3: "love",
    4: "desire",
    5: "optimism",
    6: "caring",
    7: "pride",
    8: "admiration",
    9: "gratitude",
    10: "relief",
    11: "approval",
    12: "realization",
    13: "surprise",
    14: "curiosity",
    15: "confusion",
    16: "fear",
    17: "nervousness",
    18: "remorse",
    19: "embarrassment",
    20: "disappointment",
    21: "sadness",
    22: "grief",
    23: "disgust",
    24: "anger",
    25: "annoyance",
    26: "disapproval",
    27: "neutral"
}

In [9]:
DATA_DIR = "."

train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
val_df = pd.read_csv(f"{DATA_DIR}/val.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")

train_df["labels"] = train_df["labels"].apply(ast.literal_eval)
val_df["labels"] = val_df["labels"].apply(ast.literal_eval)
test_df["labels"] = test_df["labels"].apply(ast.literal_eval)

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

X_train = train_df["text"].fillna("")
X_val = val_df["text"].fillna("")
X_test = test_df["text"].fillna("")

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

In [11]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

mlb = MultiLabelBinarizer(
    classes=list(range(28))
)

y_train = mlb.fit_transform(
    train_df["labels"]
)

y_val = mlb.transform(
    val_df["labels"]
)

y_test = mlb.transform(
    test_df["labels"]
)

print(y_train.shape)

(16531, 28)


Train baseline

In [12]:
baseline_model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    )
)

baseline_model.fit(
    X_train_tfidf,
    y_train
)

,estimator,LogisticRegre...max_iter=1000)
,n_jobs,None
,verbose,0
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None


In [13]:
y_val_prob = baseline_model.predict_proba(
    X_val_tfidf
)

threshold = 0.6

y_val_pred = (
    y_val_prob >= threshold
).astype(int)

Evaluate

In [14]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

print(
    "Precision:",
    precision_score(
        y_val,
        y_val_pred,
        average="macro",
        zero_division=0
    )
)

print(
    "Recall:",
    recall_score(
        y_val,
        y_val_pred,
        average="macro",
        zero_division=0
    )
)

print(
    "F1:",
    f1_score(
        y_val,
        y_val_pred,
        average="macro",
        zero_division=0
    )
)

Precision: 0.4754600008638888
Recall: 0.5102935315624663
F1: 0.4901130845760887


In [15]:
y_test_prob = baseline_model.predict_proba(
    X_test_tfidf
)

y_test_pred = (
    y_test_prob >= threshold
).astype(int)

In [16]:
print(
    classification_report(
        y_test,
        y_test_pred,
        target_names=[
            label_to_emotion[i]
            for i in range(28)
        ],
        zero_division=0
    )
)

                precision    recall  f1-score   support

     amusement       0.50      0.45      0.47       374
    excitement       0.37      0.41      0.39        98
           joy       0.54      0.50      0.52       204
          love       0.52      0.61      0.56       143
        desire       0.36      0.50      0.42        80
      optimism       0.57      0.76      0.65       142
        caring       0.54      0.64      0.58       150
         pride       0.71      0.64      0.67        86
    admiration       0.48      0.55      0.51       101
     gratitude       0.79      0.84      0.82       108
        relief       0.37      0.53      0.44        60
      approval       0.51      0.56      0.53       115
   realization       0.32      0.38      0.35        95
      surprise       0.45      0.49      0.47        85
     curiosity       0.45      0.53      0.49       100
     confusion       0.33      0.37      0.35        84
          fear       0.71      0.67      0.69  

In [ ]:
y_pred_lr = (
    baseline_model.predict_proba(X_val_tfidf) >= threshold
).astype(int)

print("y_pred_lr:", y_pred_lr.shape)

In [18]:
from sklearn.svm import LinearSVC

svm_model = OneVsRestClassifier(
    LinearSVC(
        class_weight="balanced",
        max_iter=5000
    )
)

svm_model.fit(X_train_tfidf, y_train)

y_pred_svm = svm_model.predict(X_val_tfidf)

print("y_pred_svm:", y_pred_svm.shape)

y_pred_svm: (2066, 28)


In [ ]:
emotion_names = [label_to_emotion[i] for i in range(28)]

print("=" * 70)
print("LOGISTIC REGRESSION")
print("=" * 70)
print(classification_report(y_val, y_pred_lr, target_names=emotion_names, zero_division=0))

print("=" * 70)
print("LINEAR SVM")
print("=" * 70)
print(classification_report(y_val, y_pred_svm, target_names=emotion_names, zero_division=0))

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

p_lr, r_lr, f_lr, sup = precision_recall_fscore_support(
    y_val, y_pred_lr, average=None, zero_division=0
)
p_svm, r_svm, f_svm, _ = precision_recall_fscore_support(
    y_val, y_pred_svm, average=None, zero_division=0
)

per_emotion = pd.DataFrame({
    "Emotion": emotion_names,
    "Support": sup,
    "P_LogReg": p_lr,
    "R_LogReg": r_lr,
    "F1_LogReg": f_lr,
    "P_SVM": p_svm,
    "R_SVM": r_svm,
    "F1_SVM": f_svm,
})

per_emotion["F1_Diff"] = per_emotion["F1_SVM"] - per_emotion["F1_LogReg"]
per_emotion["Winner"] = np.where(
    per_emotion["F1_SVM"] > per_emotion["F1_LogReg"],
    "SVM",
    "LogReg"
)

per_emotion = per_emotion.sort_values("Support", ascending=False).reset_index(drop=True)
per_emotion.round(3)

In [ ]:
print("Number of emotions won by each model:")
print(per_emotion["Winner"].value_counts())

In [ ]:
rows = []

for name, y_pred in [
    ("TF-IDF + Logistic Regression", y_pred_lr),
    ("TF-IDF + Linear SVM", y_pred_svm),
]:
    pm, rm, fm, _ = precision_recall_fscore_support(
        y_val, y_pred, average="macro", zero_division=0
    )
    pw, rw, fw, _ = precision_recall_fscore_support(
        y_val, y_pred, average="weighted", zero_division=0
    )
    rows.append({
        "Model": name,
        "P-macro": pm,
        "R-macro": rm,
        "F1-macro": fm,
        "P-weighted": pw,
        "R-weighted": rw,
        "F1-weighted": fw,
    })

comparison = pd.DataFrame(rows).sort_values(
    "F1-macro", ascending=False
).reset_index(drop=True)

comparison.round(4)